In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Users\User\OneDrive - Clemson University\flight-delay-analytics\data\flights.db")
df = pd.read_sql("SELECT * FROM flights_raw", conn)
conn.close()

df['fl_date'] = pd.to_datetime(df['fl_date'])
df['month'] = df['fl_date'].dt.month
df['day_of_week'] = df['fl_date'].dt.dayofweek  # 0=Mon
df['dep_hour'] = (df['dep_time'] // 100).astype('Int64')  # HHMM -> hour

features = ['carrier', 'origin', 'dest', 'month', 'day_of_week', 'dep_hour', 'distance']

In [2]:
df['IsDelayed'] = (df['arr_delay'] > 15).astype(int)

df['IsDelayed'].value_counts(normalize=True)

IsDelayed
0    0.801756
1    0.198244
Name: proportion, dtype: float64

In [4]:
df.columns.tolist()

['year',
 'month',
 'day_of_month',
 'day_of_week',
 'fl_date',
 'op_unique_carrier',
 'op_carrier_fl_num',
 'origin',
 'origin_city_name',
 'origin_state_nm',
 'dest',
 'dest_city_name',
 'dest_state_nm',
 'crs_dep_time',
 'dep_time',
 'dep_delay',
 'taxi_out',
 'wheels_off',
 'wheels_on',
 'taxi_in',
 'crs_arr_time',
 'arr_time',
 'arr_delay',
 'cancelled',
 'cancellation_code',
 'diverted',
 'crs_elapsed_time',
 'actual_elapsed_time',
 'air_time',
 'distance',
 'carrier_delay',
 'weather_delay',
 'nas_delay',
 'security_delay',
 'late_aircraft_delay',
 'dep_hour',
 'IsDelayed']

In [5]:
features = ['op_unique_carrier', 'origin', 'dest', 'month', 'day_of_week', 'dep_hour', 'distance']

df_model = df[features + ['IsDelayed']].dropna()

df_encoded = pd.get_dummies(df_model, columns=['op_unique_carrier', 'origin', 'dest'], drop_first=True)

In [6]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns='IsDelayed')
y = df_encoded['IsDelayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(5589137, 712) (1397285, 712)


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
y_pred = rf.predict(X_test)

In [9]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['OnTime', 'Delayed']))

              precision    recall  f1-score   support

      OnTime       0.80      1.00      0.89   1116607
     Delayed       0.00      0.00      0.00    280678

    accuracy                           0.80   1397285
   macro avg       0.40      0.50      0.44   1397285
weighted avg       0.64      0.80      0.71   1397285



C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [10]:
import joblib

joblib.dump(rf, r"C:\Users\User\OneDrive - Clemson University\flight-delay-analytics\rf_delay_model.joblib")

['C:\\Users\\User\\OneDrive - Clemson University\\flight-delay-analytics\\rf_delay_model.joblib']

In [11]:
report = classification_report(y_test, y_pred, target_names=['OnTime', 'Delayed'])

with open(r"C:\Users\User\OneDrive - Clemson University\flight-delay-analytics\week3_rf_metrics.txt", "w") as f:
    f.write(report)

C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
